# Music Annotation Project V5: Reconstruction Benchmark and Failure Analysis

**Runtime:** CPU.

This notebook measures reconstruction quality on pages for which a clean source exists. It creates synthetic occlusions using real corrected annotation masks, reconstructs the hidden clean notation, and reports pixel, edge and staff-line recovery metrics. It does not treat a visually plausible invention as ground truth.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib scikit-image


In [ ]:
from pathlib import Path
import cv2,json,math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CLEAN_DIR=ROOT/'source'/'clean'
ANNOTATED_DIR=ROOT/'aligned'
MASK_DIR=ROOT/'masks'/'corrected'
RECON_DIR=ROOT/'reconstruction'/'cleaned'
OUT=ROOT/'reconstruction_benchmark';OUT.mkdir(parents=True,exist_ok=True)
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff'}
def images(folder): return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}
clean=images(CLEAN_DIR);annotated=images(ANNOTATED_DIR);masks=images(MASK_DIR);recon=images(RECON_DIR)
pages=sorted(set(clean)&set(masks))
if not pages: raise RuntimeError('No matched clean pages and corrected masks found.')
print('Benchmark pages:',len(pages))


## Metrics

- masked MAE and PSNR measure greyscale fidelity only inside removed annotations;
- masked SSIM measures local structural similarity;
- edge F1 checks whether notation edges were restored;
- staff continuity compares horizontal ink continuity through masked areas;
- outside-mask drift confirms that untouched notation was not changed.


In [ ]:
def edge_map(im): return cv2.Canny(im,70,170)>0
def metrics(clean_img,reconstructed,mask):
    m=mask>0
    if not np.any(m): return None
    diff=np.abs(clean_img.astype(np.float32)-reconstructed.astype(np.float32))
    mae=float(diff[m].mean());mse=float(np.mean((clean_img[m].astype(np.float32)-reconstructed[m].astype(np.float32))**2))
    psnr=float(20*np.log10(255.0/math.sqrt(max(mse,1e-9))))
    score,map_=ssim(clean_img,reconstructed,data_range=255,full=True)
    masked_ssim=float(map_[m].mean())
    ce=edge_map(clean_img)&m;re=edge_map(reconstructed)&m
    tp=int(np.sum(ce&re));precision=tp/max(int(re.sum()),1);recall=tp/max(int(ce.sum()),1);edge_f1=2*precision*recall/max(precision+recall,1e-9)
    outside=float(diff[~m].mean()) if np.any(~m) else 0.0
    horizontal=cv2.morphologyEx((clean_img<180).astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,25),np.uint8))>0
    staff_zone=horizontal&m
    staff_recall=float(np.sum((reconstructed<180)&staff_zone)/max(np.sum(staff_zone),1))
    return {'masked_mae':mae,'masked_psnr':psnr,'masked_ssim':masked_ssim,'edge_f1':edge_f1,'staff_ink_recall':staff_recall,'outside_mask_drift':outside,'masked_pixels':int(m.sum())}


## Evaluate existing reconstruction outputs and simple baselines


In [ ]:
rows=[];examples=[]
for page in pages:
    c=cv2.imread(str(clean[page]),0);m=cv2.imread(str(masks[page]),0)
    if c is None or m is None or c.shape!=m.shape: continue
    # Synthetic observed page: clean notation covered with locally sampled annotation-like dark strokes.
    observed=c.copy(); observed[m>0]=np.minimum(observed[m>0],70)
    methods={'telea':cv2.inpaint(observed,m,3,cv2.INPAINT_TELEA),'navier_stokes':cv2.inpaint(observed,m,5,cv2.INPAINT_NS)}
    if page in recon:
        r=cv2.imread(str(recon[page]),0)
        if r is not None and r.shape==c.shape: methods['pipeline_output']=r
    for name,r in methods.items():
        met=metrics(c,r,m)
        if met: rows.append({'page':page,'method':name,**met})
    if len(examples)<12: examples.append((page,c,observed,methods))
df=pd.DataFrame(rows);df.to_csv(OUT/'reconstruction_metrics.csv',index=False)
summary=df.groupby('method').agg(masked_mae=('masked_mae','mean'),masked_psnr=('masked_psnr','mean'),masked_ssim=('masked_ssim','mean'),edge_f1=('edge_f1','mean'),staff_ink_recall=('staff_ink_recall','mean'),outside_mask_drift=('outside_mask_drift','mean'),pages=('page','nunique')).reset_index()
summary.to_csv(OUT/'reconstruction_summary.csv',index=False);display(summary.sort_values(['edge_f1','staff_ink_recall'],ascending=False))


## Failure-focused contact sheets


In [ ]:
if not df.empty:
    worst=df[df.method==df.method.unique()[0]].sort_values(['edge_f1','staff_ink_recall']).head(12).page.tolist()
else: worst=[]
for page,c,observed,methods in examples:
    if worst and page not in worst: continue
    panels=[c,observed]+list(methods.values());names=['clean ground truth','synthetic occlusion']+list(methods.keys())
    plt.figure(figsize=(5*len(panels),8))
    for i,(im,name) in enumerate(zip(panels,names),1):
        plt.subplot(1,len(panels),i);plt.imshow(im,cmap='gray',vmin=0,vmax=255);plt.title(name);plt.axis('off')
    plt.suptitle(page);plt.tight_layout();plt.savefig(OUT/f'{page}_comparison.png',dpi=150,bbox_inches='tight');plt.show()
print('Metrics:',OUT/'reconstruction_metrics.csv')
print('Summary:',OUT/'reconstruction_summary.csv')
